# Multi-Vector RAG

> Notebook generated from [`examples/rag/08_multi_vector_rag.py`](../../examples/rag/08_multi_vector_rag.py).

| Data | Value |
|------|-------|
| **Dataset** | arXiv (local CSV — ML/AI) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Index per doc: chunk + summary + N hypothetical questions; recall@k for the three indices.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Multi-Vector RAG — Three representations per document
======================================================
Architecture: SPEC-RAG-007 / prismal.rag.multi_vector

Dataset: ArXiv Papers (Machine Learning & AI)
  • 2.2M+ scientific papers from ArXiv across multiple domains.
  • Reference: https://huggingface.co/datasets/ccdv/arxiv-summarization
  • Why: Scientific papers have a natural mismatch between the user's query
    and the content: the user may ask with a simple concept,
    but the paper uses specialized technical vocabulary.
    Multi-Vector RAG indexes the same document under 3 representations
    (chunk, summary, hypothetical questions) to capture different
    angles of access to the document.

Multi-Vector RAG architecture description:
  For each document, generates and indexes 3 representations:
  1. Chunk    : Fragmented original text (~500 chars)
  2. Summary  : LLM-generated summary of the chunk
  3. Questions: N hypothetical questions the chunk would answer

  Search:
  - Searches across all 3 representations simultaneously
  - Deduplicates by doc_id (same document under different representations)
  - Returns the original chunk (not the representation that matched)

  Benefit: A document can be found by:
  - Its exact content (chunk)
  - A summarized concept (summary)
  - A specific question it answers (questions)

Uso:
    uv run python examples/rag/08_multi_vector_rag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import tempfile
from pathlib import Path

from prismal.rag.multi_vector import MultiVectorRAGEngine, MultiVectorResult
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: ArXiv paper abstracts and fragments

In [ ]:
# Representative ML/AI papers from ArXiv (2023-2024).
ARXIV_PAPERS = [
    {
        "arxiv_id": "2302.04761",
        "title": "Language Is Not All You Need: Aligning Perception with Language Models",
        "filename": "kosmos1_multimodal.txt",
        "content": (
            "We introduce KOSMOS-1, a Multimodal Large Language Model (MLLM) that can perceive "
            "general modalities, learn in context (i.e., few-shot), and follow instructions "
            "(i.e., zero-shot). Specifically, we train KOSMOS-1 from scratch on web-scale "
            "multimodal corpora, including arbitrarily interleaved text and images, image-caption "
            "pairs, and text data. We evaluate various settings, including zero-shot, few-shot, "
            "and multimodal chain-of-thought prompting. KOSMOS-1 achieves impressive performance "
            "on language understanding, generation, OCR-free NLP (e.g., FLAN tokens), "
            "perception-language tasks (e.g., image captioning, visual QA), speech recognition, "
            "and speech-to-text translation. "
            "A key capability of MLLMs is cross-modal transfer, where knowledge learned from one "
            "modality can be applied to another. KOSMOS-1 demonstrates this by achieving "
            "competitive results on visual commonsense reasoning tasks despite being trained "
            "primarily on language data. The model architecture uses a transformer backbone "
            "with vision encoders that process image patches as tokens alongside text tokens."
        ),
    },
    {
        "arxiv_id": "2307.09288",
        "title": "Llama 2: Open Foundation and Fine-Tuned Chat Models",
        "filename": "llama2_paper.txt",
        "content": (
            "In this work, we develop and release Llama 2, a collection of pretrained and "
            "fine-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion "
            "parameters. Our fine-tuned LLMs, called Llama 2-Chat, are optimized for dialogue use "
            "cases. Our models outperform open-source chat models on most benchmarks we tested, "
            "and based on our human evaluations for helpfulness and safety, may be a suitable "
            "substitute for closed-source models. We provide a detailed description of our "
            "approach to fine-tuning and safety improvements of Llama 2-Chat in order to enable "
            "the community to build on our work and contribute to the responsible development of LLMs. "
            "We used Reinforcement Learning from Human Feedback (RLHF) with Proximal Policy "
            "Optimization (PPO) and rejection sampling to align the models with human preferences. "
            "Ghost Attention (GAtt) was introduced to maintain instruction following in multi-turn "
            "conversations by conditioning on the system prompt throughout the dialogue."
        ),
    },
    {
        "arxiv_id": "2310.11511",
        "title": "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection",
        "filename": "self_rag_paper.txt",
        "content": (
            "We introduce Self-RAG, a framework that trains an arbitrary LM to reflectively "
            "retrieve passages on demand, and generate and critique its own output. Unlike "
            "conventional RAG that always retrieves a fixed number of passages regardless of "
            "whether retrieval is necessary, Self-RAG trains a single arbitrary LM that "
            "adaptively retrieves passages on demand, and generates and reflects on retrieved "
            "passages and its own generations using special tokens, called reflection tokens. "
            "Generating reflection tokens makes the LM controllable during the inference phase, "
            "enabling it to tailor its behavior to diverse task requirements. Experiments show "
            "that Self-RAG (7B and 13B parameters) significantly outperforms state-of-the-art "
            "LLMs and retrieval-augmented models on various tasks including open-domain QA, "
            "reasoning, and fact verification. Self-RAG introduces four types of special tokens: "
            "[Retrieve], [IsREL], [IsSUP], and [IsUSE] for controlling retrieval, relevance "
            "grading, support assessment, and utility scoring respectively."
        ),
    },
    {
        "arxiv_id": "2310.06825",
        "title": "Mistral 7B",
        "filename": "mistral7b_paper.txt",
        "content": (
            "We introduce Mistral 7B, a 7-billion-parameter language model engineered for "
            "superior performance and efficiency. Mistral 7B outperforms the best open 13B model "
            "(Llama 2 13B) across all evaluated benchmarks, and the best released 34B model "
            "(Llama 1 34B) in reasoning, mathematics, and code generation. Our model leverages "
            "grouped-query attention (GQA) for faster inference, coupled with sliding window "
            "attention (SWA) to effectively handle sequences of arbitrary length with a reduced "
            "inference cost. We also provide a model fine-tuned to follow instructions, Mistral "
            "7B-Instruct, that surpasses Llama 2 13B-chat on both human and automated benchmarks. "
            "The sliding window attention with a window size of 4096 allows the model to process "
            "sequences up to 128K tokens during inference through a rolling buffer KV cache mechanism."
        ),
    },
    {
        "arxiv_id": "2401.00368",
        "title": "Improving Text Embeddings with Large Language Models",
        "filename": "llm_embeddings_paper.txt",
        "content": (
            "We introduce a novel and simple approach for obtaining high-quality text embeddings "
            "using only synthetic data and less than 1K training steps. Unlike prior work that "
            "trains on expensive human-labeled data or relies on complex training pipelines, "
            "our approach generates diverse synthetic data for hundreds of thousands of text "
            "embedding tasks across 93 languages. We then fine-tune an LLM on the synthetic "
            "data following standard contrastive learning with in-batch negatives. Experiments "
            "demonstrate that our method achieves strong performance on the Massive Text Embedding "
            "Benchmark (MTEB) and BEIR retrieval benchmarks. The key insight is that LLMs can "
            "generate high-quality diverse training examples by following simple prompts, "
            "eliminating the need for expensive human annotation in embedding model training."
        ),
    },
]

## Queries with different match types

In [ ]:
MULTIVECTOR_QUERIES = [
    {
        "id": "MVQ1",
        "query": "How do multimodal LLMs handle visual perception?",
        "expected_source_prefix": "kosmos1",
        "match_type": "summary",
        "description": "Match via summary (conceptual) — the text uses different vocabulary",
    },
    {
        "id": "MVQ2",
        "query": "What technique does Llama 2 use for alignment with human preferences?",
        "expected_source_prefix": "llama2",
        "match_type": "question",
        "description": "Match via hypothetical question — the paper answers exactly this",
    },
    {
        "id": "MVQ3",
        "query": "reflection tokens RETRIEVE IsSUP IsUSE special tokens self-reflection",
        "expected_source_prefix": "self_rag",
        "match_type": "chunk",
        "description": "Match via direct chunk — exact technical keywords",
    },
    {
        "id": "MVQ4",
        "query": "What 7B model beats 13B models on reasoning benchmarks?",
        "expected_source_prefix": "mistral",
        "match_type": "question",
        "description": "Direct question about a benchmark — match via hypothetical questions",
    },
    {
        "id": "MVQ5",
        "query": "training embeddings with synthetic data without human annotation",
        "expected_source_prefix": "llm_embeddings",
        "match_type": "summary",
        "description": "Key summarized concept — match via summary",
    },
]


async def setup_multivector_rag() -> MultiVectorRAGEngine:
    """Create and initialize the Multi-Vector RAG engine with ArXiv papers.

    Returns:
        MultiVectorRAGEngine with the 5 papers indexed.
    """
    engine = MultiVectorRAGEngine(
        vector_store=ChromaVectorStore(collection_name="arxiv_multivector"),
        n_questions=3,  # generate 3 hypothetical questions per chunk
    )

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)

        for paper in ARXIV_PAPERS:
            paper_path = tmpdir_path / paper["filename"]
            # Include the title in the content for better context
            full_content = f"Title: {paper['title']}\n\nAbstract:\n{paper['content']}"
            paper_path.write_text(full_content, encoding="utf-8")
            await engine.index_document(paper_path)
            print(f"  ✓ Indexed: {paper['filename']} (3 representations)")

    return engine


def print_multivector_result(
    query_info: dict,
    result: MultiVectorResult,
) -> None:
    """Show the Multi-Vector results along with which representation matched."""
    expected_prefix = query_info["expected_source_prefix"]

    print(f"\n[{query_info['id']}] {query_info['query']}")
    print(f"  Expected match type: {query_info['match_type']}")
    print(f"  Description: {query_info['description']}")

    print("\n  Top-3 results:")
    for i, chunk in enumerate(result.chunks[:3], 1):
        expected = expected_prefix in chunk.source
        mark = "→" if expected else " "
        print(
            f"    {mark}[{i}] [{chunk.relevance_score:.3f}] {chunk.source}: {chunk.content[:80]}..."
        )

    # Show which representations matched
    if result.matched_representations:
        print("\n  Representations that matched:")
        for doc_id, representations in list(result.matched_representations.items())[:3]:
            print(f"    {doc_id}: {representations}")

    # Verify whether the expected document is in the results
    found = any(expected_prefix in c.source for c in result.chunks[:3])
    print(f"\n  Expected document found: {'✓' if found else '✗'}")
    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  Multi-Vector RAG — Dataset: ArXiv Papers (ML/AI)")
    print("=" * 70)

    # Architecture
    print("\n[Multi-Vector architecture]")
    print("  For each document, 3 representations are generated and indexed:")
    print()
    print("  Document")
    print("  ├── [Chunk]     Fragmented original text")
    print("  │                → Indexed with: embed(chunk_text)")
    print("  ├── [Summary]   LLM summary of the chunk")
    print("  │                → Indexed with: embed(summary)")
    print("  └── [Questions] N hypothetical questions the chunk answers")
    print("                   → Indexed with: embed(question_i)")
    print()
    print("  All share the same doc_id → dedup in results")

    # Initialization
    print("\n[Indexing ArXiv papers]")
    print("  (Each paper generates ~3 chunks × 3 representations = ~9 vectors)")
    engine = await setup_multivector_rag()
    total_vectors = len(ARXIV_PAPERS) * 3 * 3
    print(f"  ✓ ~{total_vectors} vectors indexed for {len(ARXIV_PAPERS)} papers")

    # Run queries
    print(f"\n[Running {len(MULTIVECTOR_QUERIES)} queries over ArXiv papers]")

    correct_count = 0
    match_type_stats: dict[str, int] = {"chunk": 0, "summary": 0, "question": 0}

    for query_info in MULTIVECTOR_QUERIES:
        result = await engine.search(query_info["query"], k=5)
        print_multivector_result(query_info, result)

        expected_prefix = query_info["expected_source_prefix"]
        if any(expected_prefix in c.source for c in result.chunks[:3]):
            correct_count += 1
            match_type_stats[query_info["match_type"]] = (
                match_type_stats.get(query_info["match_type"], 0) + 1
            )

    # Summary
    recall = correct_count / len(MULTIVECTOR_QUERIES)
    print("\n[Statistical summary]")
    print(f"  Recall@3: {correct_count}/{len(MULTIVECTOR_QUERIES)} ({recall:.0%})")

    # Representation comparison
    print("\n[Match type distribution]")
    print("  Which representations are most useful:")
    for rep_type, desc in [
        ("chunk", "exact technical keywords from the paper"),
        ("summary", "general concepts and themes"),
        ("question", "direct questions about the content"),
    ]:
        count = match_type_stats.get(rep_type, 0)
        print(f"    {rep_type:10s}: {count} queries ← {desc}")

    # Multi-Vector vs Chunk-only comparison
    print("\n[Multi-Vector vs standard RAG (chunks only)]")
    comparison = [
        ("Multi-Vector", "High", "High", "3×"),
        ("Standard RAG", "Medium", "Medium", "1×"),
    ]
    print(f"  {'Method':<15} {'Recall':<8} {'Precision':<10} {'Indexing cost':>18}")
    print("  " + "─" * 53)
    for method, recall_q, precision, cost in comparison:
        print(f"  {method:<15} {recall_q:<8} {precision:<10} {cost:>18}")

    print("\n[When to use Multi-Vector RAG]")
    print("  ✓ Technical documents with specialized vocabulary")
    print("  ✓ Users who ask in different styles/languages")
    print("  ✓ Corpora of scientific papers or complex documentation")
    print("  ✗ Very large corpus (3× vectors = 3× indexing cost)")
    print("  ✗ Simple/short documents where a single chunk is enough")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()